In [15]:
import os
import pandas as pd
import pickle
from dotenv import load_dotenv
from groq import Groq
import utils.llm as llm   # kept for parse_all_scores / parse_score helpers
import importlib
import time
from collections import deque

In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    api_key        = os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint = os.getenv("AZURE_AZURE_ENDPOINT"),
    api_version    = "2024-12-01-preview",
)

In [ ]:
# Add or remove models here.
# key   → short name used for column suffixes (e.g. mar_score_kimi_0)
# value → full Groq model ID
MODELS = {
    "gpt-4o": "gpt-4o",
}

RPM = {"gpt-4o": 10}

In [ ]:
dr_masked = pd.read_pickle('data/dr/text/leakage/dr_masked.pkl')
dr_masked

## Score dict

In [19]:
score_dict_2 = {
    "sty": "Style, precision, understandability, professionality and appropriateness of the text - 1: low quality of the text, stilistic issues, 10: well structured, fluent text with a guiding thread and no major errors",
    "tec": "Technical knowledge of the author - 1: author seems rather clueless in terms of technical details, 10: author knows all relevant technical specifics",
    "mak": "Market knowledge of the author - 1: the author doesn't adress market factors at all or very vaguely/incorrectly, 10: the author shows high market awareness and has a strategic approach towards product-market fit",
    "dow": "Down-to-earthness of the author - 1: extreme exaggerations and buzzword usage, 10: very down-to earth tone, realistic scenarios, no empty phrases and buzz words",
    "mar": "Market focus and clarity of the business case presented in the text - 1: no adressing of the market or unrealistic expectations, 10: high market awareness, strategic plan on marketing the product or service",
    "nov": "Novelty/originality of the business model presented in the text - 1: generic business idea that has been seen a lot, 10: novel and original business/product idea",
    "via": "Operational viability of the business model presented in the text - 1: unrealistic operational expecations and assumptions, 10: differentiated and nuanced business idea",
    "atr": "Attractiveness of the product or service presented in the text - 1: product or service that has little or no use for the customer, in relation to the expected price, 10: high utility of the product or service in relation to the price"
}

## generate_scores

In [ ]:
def generate_scores(pitch_text, score_dict, model="kimi-k2.5", temperature=0.0, max_tokens=1000, calls_per_min=10):
    """
    Calls the Groq API and returns:
        { score_code: {"score": float|None, "response": str} }
    Compatible with llm.score_df as generate_scores_func.
    """
    results = {}

    if pitch_text is None or pd.isna(pitch_text) or str(pitch_text).strip().lower() in ('nan', ''):
        return None  # instead of returning empty score dicts

    system_prompt = (
        "You are a critical VC investor tasked with evaluating a startup pitch. "
        "Be very critical and only refer to the criteria provided. "
        "Utilize the full range of scores 1-10."
    )

    user_prompt = f"""START OF THE PITCH TEXT >>>
{pitch_text}
<<< END OF THE PITCH TEXT

Score the startup pitch above for the following dimensions:
{score_dict}

Provide a score out of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] and a very short sentence for reasoning.

Return the score EXACTLY in this format (one line per dimension):
sty: X - Reason: [short reasoning]
tec: X - Reason: [short reasoning]
mak: X - Reason: [short reasoning]
dow: X - Reason: [short reasoning]
mar: X - Reason: [short reasoning]
nov: X - Reason: [short reasoning]
via: X - Reason: [short reasoning]
atr: X - Reason: [short reasoning]
"""

    # ── rate limiter: max 6 calls / 60 s (sliding window) ──────────────────
    _now = time.monotonic()
    if not hasattr(generate_scores, "_call_times"):
        generate_scores._call_times = deque()
    _ct = generate_scores._call_times
    while _ct and _now - _ct[0] > 60:
        _ct.popleft()
    if len(_ct) >= calls_per_min:
        _sleep = 60 - (_now - _ct[0]) + 0.1
        #print(f"Rate limit: sleeping {_sleep:.1f}s")
        time.sleep(_sleep)
        _now = time.monotonic()
        while _ct and _now - _ct[0] > 60:
            _ct.popleft()
    generate_scores._call_times.append(time.monotonic())
    # ────────────────────────────────────────────────────────────────────────

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=temperature,
            max_completion_tokens=max_tokens,
            stream=False,
        )
        full_response = response.choices[0].message.content.strip()

    except Exception as e:
        err = str(e).lower()
        if any(phrase in err for phrase in ("rate_limit_exceeded", "limit reached", "tokens per day", "billing", "quota")):
            raise RuntimeError(f"Limit hit — stopping.\nOriginal error: {e}") from e
        print(f"Call failed ({model}): {e}")
        return None  # also return None instead of empty results, so _process_row_score skips the row

    parsed_scores = llm.parse_all_scores(full_response, list(score_dict.keys()))

    for code in score_dict:
        if code in parsed_scores:
            results[code] = {
                "score":    parsed_scores[code]["score"],
                "response": parsed_scores[code]["line"],
            }
        else:
            results[code] = {"score": None, "response": ""}

    return results

## Quick single-row test

In [ ]:
test_text = """Test Text"""

result = generate_scores(test_text, score_dict_2)
for code, val in result.items():
    print(f"{code}: {val['response']}")

sty: sty: 6 - Reason: The text is professional and understandable but lacks depth and a clear guiding thread.
tec: tec: 4 - Reason: The pitch mentions AI and gamification but provides no technical specifics or evidence of expertise.
mak: mak: 5 - Reason: The market is vaguely addressed, with no detailed analysis or strategic insights.
dow: dow: 4 - Reason: The pitch relies heavily on buzzwords like "gamification" and "AI" without substantiating claims.
mar: mar: 4 - Reason: The business case is unclear, with no mention of target customers, pricing, or go-to-market strategy.
nov: nov: 3 - Reason: Workforce Performance Management with gamification and AI is not a novel concept.
via: via: 4 - Reason: The operational viability is questionable due to the lack of detail on execution or differentiation.
atr: atr: 5 - Reason: The product could be useful, but its value proposition and pricing are not clearly articulated.


## Score full DataFrame — all models

In [ ]:
results_all = {}

for short_name, model_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Running: {short_name}  ({model_id})")
    print('='*60)

    checkpoint_path = f"data/dr/text/scoring/azure/{short_name}_temp0_scores_final.pkl"
    try:
        results_all[short_name] = llm.score_df_api(
            df                   = dr_masked,
            score_dict           = score_dict_2,
            text_col             = "masked",
            generate_scores_func = generate_scores,
            model                = model_id,
            generate_kwargs      = {"temperature": 0.0, "calls_per_min": RPM[short_name]},
            workers              = 1,
            checkpoint_file      = checkpoint_path,
            save_frequency       = 5,
        )
    except Exception as e:
            print(f"\n Model {short_name} stopped")
print("\nAll models finished.")